# Regressione Logistica: Esercizi in classe (scikit-learn)

Questo notebook contiene una serie di esercizi progressivi su:
- `predict_proba` e interpretazione delle probabilità
- soglia (threshold) e come cambia `y_pred`
- confusion matrix, accuracy, precision, recall
- log-loss (cross-entropy) e perché **non cambia** quando cambi la soglia

> Suggerimento docente: fai eseguire le celle **una alla volta**.  
> Gli studenti devono completare le parti marcate con `# TODO`.

---


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, log_loss, precision_score, recall_score

%matplotlib inline


## Funzioni di supporto (non modificare)

Useremo queste funzioni per plottare dati e curve.


In [ ]:
def plot_1d_data(x, y, ax=None):
    if ax is None:
        fig, ax = plt.subplots(figsize=(6,2.2))
    x = np.ravel(x)
    y = np.ravel(y)
    ax.scatter(x[y==0], y[y==0], s=110, facecolors="none", edgecolors="black", label="y=0")
    ax.scatter(x[y==1], y[y==1], s=110, marker="x", label="y=1")
    ax.set_ylim(-0.2, 1.2)
    ax.set_xlabel("x")
    ax.set_ylabel("y")
    ax.grid(True, alpha=0.3)
    ax.legend()
    return ax

def plot_sigmoid_curve(model, x, ax=None, x_pad=0.5, title="P(y=1|x)"):
    if ax is None:
        fig, ax = plt.subplots(figsize=(7,3))
    x = np.ravel(x)
    x_line = np.linspace(x.min()-x_pad, x.max()+x_pad, 400)
    X_line = x_line.reshape(-1,1)
    p_line = model.predict_proba(X_line)[:,1]
    ax.plot(x_line, p_line, linewidth=2, label="P(y=1|x)")
    ax.axhline(0.5, linewidth=1, label="soglia 0.5")
    ax.set_ylim(-0.05, 1.05)
    ax.set_xlabel("x")
    ax.set_ylabel("probabilità")
    ax.set_title(title)
    ax.grid(True, alpha=0.3)
    ax.legend()
    return ax

def logistic_loss_single(y, p, eps=1e-15):
    p = np.clip(p, eps, 1-eps)  # evita log(0)
    return -(y*np.log(p) + (1-y)*np.log(1-p))


## Dataset di partenza (1 variabile)

Useremo un dataset semplice per vedere chiaramente la sigmoide.

In [ ]:
x = np.array([0., 1., 2., 3., 4., 5.])
y = np.array([0,   0,  0,  1,  1,  1])

X = x.reshape(-1,1)

ax = plot_1d_data(x, y)
plt.show()


## Esercizio 1 — Addestra un modello e leggi `predict_proba`

1) Addestra `LogisticRegression()` sul dataset.  
2) Stampa `predict_proba(X)` e rispondi:
- Quante colonne ci sono?
- Cosa rappresenta la colonna 0?
- Cosa rappresenta la colonna 1?
- La somma sulle due colonne vale sempre 1?


In [ ]:
# TODO: crea e addestra il modello
model = LogisticRegression()
model.fit(X, y)

proba = model.predict_proba(X)
print("predict_proba(X):\n", np.round(proba, 3))

# TODO: verifica che ogni riga sommi a 1
print("\nSomma righe:", np.round(proba.sum(axis=1), 3))


## Esercizio 2 — Disegna la curva delle probabilità

1) Disegna la curva P(y=1|x) usando `plot_sigmoid_curve(model, x)`  
2) Individua “a occhio” dove la curva attraversa 0.5 (decision boundary).


In [ ]:
ax = plot_sigmoid_curve(model, x, title="Regressione logistica: probabilità stimata")
plt.show()


## Esercizio 3 — Calcola la decision boundary con w e b (caso 1D)

Per una sola variabile, lo score è: `z = w*x + b`.  
La decision boundary (soglia 0.5) è dove `z=0`, quindi:

`x* = -b / w`

1) Estrai `w` e `b` dal modello (`model.coef_`, `model.intercept_`)  
2) Calcola `x*` e stampalo.


In [ ]:
w = float(model.coef_[0][0])
b = float(model.intercept_[0])

x_star = -b / w
print(f"w = {w:.4f}")
print(f"b = {b:.4f}")
print(f"Decision boundary x* = {x_star:.4f}  (dove P=0.5)")


## Esercizio 4 — Cambia la soglia (threshold) e confronta la confusion matrix

Importante:
- `model.predict(X)` usa **sempre** soglia 0.5 internamente.
- Se vuoi cambiare soglia devi usare `predict_proba` e poi applicare tu la soglia.

Fai così:
1) Calcola `p_train = model.predict_proba(X)[:,1]`
2) Scegli una soglia `threshold` (es. 0.7)
3) Crea `y_pred_thr = (p_train >= threshold).astype(int)`
4) Confronta confusion matrix e accuracy per soglia 0.5 vs 0.7


In [ ]:
p_train = model.predict_proba(X)[:,1]

# TODO: cambia questa soglia (prova 0.3, 0.5, 0.7)
threshold = 0.7

y_pred_05 = (p_train >= 0.5).astype(int)
y_pred_thr = (p_train >= threshold).astype(int)

print("Probabilità (p_train):", np.round(p_train, 3))

print("\n--- Soglia 0.5 ---")
print("y_pred:", y_pred_05)
print("Confusion matrix:\n", confusion_matrix(y, y_pred_05))
print("Accuracy:", accuracy_score(y, y_pred_05))

print(f"\n--- Soglia {threshold} ---")
print("y_pred:", y_pred_thr)
print("Confusion matrix:\n", confusion_matrix(y, y_pred_thr))
print("Accuracy:", accuracy_score(y, y_pred_thr))


<details><summary><b>Nota importante</b></summary>

La soglia cambia **y_pred**, quindi cambiano confusion matrix e accuracy.  
</details>


## Esercizio 5 — Perché la log-loss non cambia quando cambi soglia?

Calcola:
- `log_loss(y, p_train)`  (usa le probabilità)
- `log_loss(y, y_pred_thr)` (⚠️ sbagliato concettualmente: qui non stai passando probabilità)

Poi rispondi:
1) Perché `log_loss(y, p_train)` resta uguale se cambi threshold?
2) Quale valore è quello corretto da usare per log-loss?


In [ ]:
print("Log-loss con probabilità (corretto):", log_loss(y, p_train))

# Questo può anche dare errore o risultare poco sensato: stai passando classi 0/1 invece di probabilità
try:
    print("Log-loss con classi 0/1 (non consigliato):", log_loss(y, y_pred_thr))
except Exception as e:
    print("Log-loss con classi 0/1 -> ERRORE (atteso):", e)


<details><summary><b>Soluzione (docente)</b></summary>

La log-loss usa **pi** (probabilità) nella formula.  
Cambiare la soglia cambia solo la conversione `pi -> y_pred`, ma non cambia `pi`.  
Quindi `log_loss(y, p_train)` resta uguale finché `p_train` è lo stesso.
</details>


## Esercizio 6 — Accuracy vs Log-loss possono “raccontare cose diverse”

Costruiamo due set di probabilità:

- Caso A: probabilità “conservative” (vicine a 0.5)
- Caso B: probabilità “molto sicure” ma con un errore grave su un punto

Confronta accuracy e log-loss nei due casi.


In [ ]:
y_true = np.array([0,0,0,1,1,1])

# Caso A: conservative
p_A = np.array([0.49, 0.45, 0.40, 0.60, 0.55, 0.51])

# Caso B: molto sicure ma un errore grave (ultimo punto: y=1 ma p=0.01)
p_B = np.array([0.01, 0.02, 0.05, 0.95, 0.90, 0.01])

def eval_probs(y_true, p, threshold=0.5):
    y_pred = (p >= threshold).astype(int)
    return {
        "threshold": threshold,
        "accuracy": accuracy_score(y_true, y_pred),
        "logloss": log_loss(y_true, p),
        "cm": confusion_matrix(y_true, y_pred)
    }

res_A = eval_probs(y_true, p_A, threshold=0.5)
res_B = eval_probs(y_true, p_B, threshold=0.5)

print("=== Caso A (conservative) ===")
print("Accuracy:", res_A["accuracy"])
print("Log-loss:", res_A["logloss"])
print("Confusion matrix:\n", res_A["cm"])

print("\n=== Caso B (sicuro ma un errore grave) ===")
print("Accuracy:", res_B["accuracy"])
print("Log-loss:", res_B["logloss"])
print("Confusion matrix:\n", res_B["cm"])


**Domande:**  
1) Quale caso ha accuracy migliore?  
2) Quale caso ha log-loss migliore?  
3) Perché la log-loss “punisce” tanto previsioni molto sicure ma sbagliate?


## Esercizio 7 (bonus) — Dataset 2D e decision boundary (retta)

Qui usiamo 2 feature (x0, x1). In 2D la decision boundary è una retta:

w0*x0 + w1*x1 + b = 0

e quindi:

x1 = -(w0/w1)*x0 - (b/w1)

Esegui e osserva la retta che separa le classi.


In [ ]:
# Dataset 2D
X2 = np.array([[0.5, 1.5],
               [1,   1],
               [1.5, 0.5],
               [3,   0.5],
               [2,   2],
               [1,   2.5]])
y2 = np.array([0, 0, 0, 1, 1, 1])

m2 = LogisticRegression()
m2.fit(X2, y2)

w0 = m2.coef_[0][0]
w1 = m2.coef_[0][1]
b2 = m2.intercept_[0]

# linea
x0_line = np.linspace(0, 4, 200)
x1_line = -(w0/w1)*x0_line - (b2/w1)

plt.figure(figsize=(5,4))
plt.scatter(X2[y2==0,0], X2[y2==0,1], marker='o', facecolors='none', edgecolors='blue', s=100, label="y=0")
plt.scatter(X2[y2==1,0], X2[y2==1,1], marker='x', color='red', s=100, label="y=1")
plt.plot(x0_line, x1_line, linewidth=2, label="decision boundary")

plt.axis([0, 4, 0, 3.5])
plt.xlabel("x0")
plt.ylabel("x1")
plt.title("Decision boundary in 2D (retta)")
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()

print(f"Decision boundary: {w0:.3f}*x0 + {w1:.3f}*x1 + {b2:.3f} = 0")
